# OR-Tools scheduling intuition

This notebook uses Google OR-Tools CP-SAT, the same solver family used by this project. The goal is not to build the whole app here. The goal is to see the shape of the model: variables, hard constraints, and optional objective terms.

Run the cells top to bottom. Then edit the small data dictionaries and rerun individual examples.

## Setup

CP-SAT models are usually built from:

- Boolean variables: choose this candidate or not.
- Integer variables: start time, end time, count, shortage, excess.
- Constraints: exactly one, at most one, no overlap, travel gaps.
- Objective: maximize preferences or minimize penalties.

In [ ]:
from collections import defaultdict
from itertools import combinations

from ortools.sat.python import cp_model


def solve(model, seconds=5):
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = seconds
    solver.parameters.num_search_workers = 8
    status = solver.Solve(model)
    print('status:', solver.StatusName(status))
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print('objective:', solver.ObjectiveValue())
    return solver, status


def clock(minutes_after_18):
    total = 18 * 60 + minutes_after_18
    return f'{total // 60:02d}:{total % 60:02d}'


def print_table(rows):
    for row in rows:
        print(' | '.join(str(value) for value in row))

## Example 1: candidate selection with Boolean variables

This is close to the current project solver. We generate every valid candidate assignment first, then ask the solver to pick a compatible set.

A candidate means: this group, in this slot, in this room, with these instructors.

The model says:

- Pick exactly one candidate for each group.
- Do not use the same room twice in one slot.
- Do not use the same teacher twice in one slot.
- Prefer some pairings and time slots.

In [ ]:
groups = {
    'Let\'s Start 1 #1': {'level': 'Let\'s Start 1', 'roles': ('leader', 'follower')},
    'Let\'s Start 1 #2': {'level': 'Let\'s Start 1', 'roles': ('leader', 'follower')},
    'Solo Jazz': {'level': 'Solo Jazz', 'roles': ('solo',)},
}

slots = ['Mon 18:00', 'Mon 19:30', 'Tue 18:00']
rooms = ['Room 1', 'Room 2']

instructors = {
    'Ania': {'roles': {'follower'}, 'levels': {'Let\'s Start 1'}, 'prefers_with': {'Mateusz'}},
    'Mateusz': {'roles': {'leader', 'solo'}, 'levels': {'Let\'s Start 1', 'Solo Jazz'}, 'prefers_with': {'Ania'}},
    'Marysia': {'roles': {'follower', 'solo'}, 'levels': {'Let\'s Start 1', 'Solo Jazz'}, 'prefers_with': set()},
}


def instructor_options(level, roles):
    eligible = [
        name
        for name, data in instructors.items()
        if level in data['levels']
    ]
    if len(roles) == 1:
        role = roles[0]
        return [(name,) for name in eligible if role in instructors[name]['roles']]

    pairs = []
    first_role, second_role = roles
    for first, second in combinations(eligible, 2):
        if first_role in instructors[first]['roles'] and second_role in instructors[second]['roles']:
            pairs.append((first, second))
        if first_role in instructors[second]['roles'] and second_role in instructors[first]['roles']:
            pairs.append((second, first))
    return pairs


candidates = []
for group_name, group in groups.items():
    for slot in slots:
        for room in rooms:
            for teachers in instructor_options(group['level'], group['roles']):
                score = 0
                if len(teachers) == 2:
                    first, second = teachers
                    score += int(second in instructors[first]['prefers_with'])
                    score += int(first in instructors[second]['prefers_with'])
                if group_name == 'Solo Jazz' and slot == 'Tue 18:00':
                    score += 2
                candidates.append({
                    'group': group_name,
                    'slot': slot,
                    'room': room,
                    'teachers': teachers,
                    'score': score,
                })

print('candidate count:', len(candidates))
candidates[:5]

In [ ]:
model = cp_model.CpModel()
x = {index: model.NewBoolVar(f'candidate_{index}') for index in range(len(candidates))}

# Exactly one assignment per group.
for group_name in groups:
    model.AddExactlyOne(x[index] for index, candidate in enumerate(candidates) if candidate['group'] == group_name)

# Room capacity: one group per room per slot.
for slot in slots:
    for room in rooms:
        model.AddAtMostOne(
            x[index]
            for index, candidate in enumerate(candidates)
            if candidate['slot'] == slot and candidate['room'] == room
        )

# Teacher capacity: one group per teacher per slot.
for slot in slots:
    for teacher in instructors:
        model.AddAtMostOne(
            x[index]
            for index, candidate in enumerate(candidates)
            if candidate['slot'] == slot and teacher in candidate['teachers']
        )

model.Maximize(sum(candidate['score'] * x[index] for index, candidate in enumerate(candidates)))

solver, status = solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    rows = []
    for index, candidate in enumerate(candidates):
        if solver.BooleanValue(x[index]):
            rows.append((candidate['slot'], candidate['room'], candidate['group'], ', '.join(candidate['teachers']), candidate['score']))
    print_table(sorted(rows))

## Example 2: interval variables and `AddNoOverlap`

The other common OR-Tools style is to model start/end times directly. Each lesson becomes an interval. Then `AddNoOverlap` prevents a resource from doing two intervals at the same time.

This is good when start times are flexible. The candidate approach is often simpler when you already have a fixed menu of legal lesson blocks.

In [ ]:
lessons = [
    {'name': 'Private A', 'duration': 60, 'teacher': 'Anna', 'room': 'Main'},
    {'name': 'Private B', 'duration': 45, 'teacher': 'Anna', 'room': 'Main'},
    {'name': 'Workshop', 'duration': 90, 'teacher': 'Mateusz', 'room': 'Main'},
    {'name': 'Solo practice', 'duration': 45, 'teacher': 'Anna', 'room': 'Side'},
]

horizon = 4 * 60
model = cp_model.CpModel()

starts = {}
ends = {}
intervals = {}
by_teacher = defaultdict(list)
by_room = defaultdict(list)

for lesson in lessons:
    name = lesson['name']
    duration = lesson['duration']
    start = model.NewIntVar(0, horizon - duration, f'{name}_start')
    end = model.NewIntVar(duration, horizon, f'{name}_end')
    interval = model.NewIntervalVar(start, duration, end, f'{name}_interval')
    starts[name] = start
    ends[name] = end
    intervals[name] = interval
    by_teacher[lesson['teacher']].append(interval)
    by_room[lesson['room']].append(interval)

for teacher, teacher_intervals in by_teacher.items():
    model.AddNoOverlap(teacher_intervals)

for room, room_intervals in by_room.items():
    model.AddNoOverlap(room_intervals)

makespan = model.NewIntVar(0, horizon, 'makespan')
for end in ends.values():
    model.Add(makespan >= end)
model.Minimize(makespan)

solver, status = solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    rows = []
    for lesson in lessons:
        name = lesson['name']
        start = solver.Value(starts[name])
        end = solver.Value(ends[name])
        rows.append((clock(start), clock(end), lesson['room'], lesson['teacher'], name))
    print_table(sorted(rows))

## Example 3: travel time between locations

This mirrors the rule in the app: the same teacher can do consecutive classes in the same location, but needs a flat 60-minute gap if the next class is in a different location.

The examples below force three lessons to be selected. Only the location and gap changes.

In [ ]:
TRAVEL_GAP = 60


def has_travel_conflict(first, second):
    if first['teacher'] != second['teacher']:
        return False
    if first['location'] == second['location']:
        return False
    gap_a_to_b = second['start'] - first['end']
    gap_b_to_a = first['start'] - second['end']
    return 0 <= gap_a_to_b < TRAVEL_GAP or 0 <= gap_b_to_a < TRAVEL_GAP


def solve_travel_case(third_location, third_start):
    fixed_lessons = [
        {'name': 'Let\'s Start 1 #1', 'teacher': 'Anna', 'location': 'Main Studio', 'start': 0, 'end': 60},
        {'name': 'Let\'s Start 1 #2', 'teacher': 'Anna', 'location': 'Main Studio', 'start': 60, 'end': 120},
        {'name': 'Let\'s Start 1 #3', 'teacher': 'Anna', 'location': third_location, 'start': third_start, 'end': third_start + 60},
    ]

    model = cp_model.CpModel()
    chosen = {lesson['name']: model.NewBoolVar(lesson['name']) for lesson in fixed_lessons}

    # Force every lesson to happen so conflicts make the model infeasible.
    for lesson in fixed_lessons:
        model.Add(chosen[lesson['name']] == 1)

    for first, second in combinations(fixed_lessons, 2):
        overlaps = first['start'] < second['end'] and second['start'] < first['end']
        same_teacher_overlap = first['teacher'] == second['teacher'] and overlaps
        if same_teacher_overlap or has_travel_conflict(first, second):
            model.Add(chosen[first['name']] + chosen[second['name']] <= 1)

    solver, status = solve(model)
    print('third lesson:', third_location, clock(third_start), '-', clock(third_start + 60))
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        rows = [(clock(item['start']), clock(item['end']), item['location'], item['teacher'], item['name']) for item in fixed_lessons]
        print_table(rows)
    print()


print('Case A: three consecutive classes in the same location')
solve_travel_case('Main Studio', 120)

print('Case B: immediate move to another location')
solve_travel_case('Riverside Studio', 120)

print('Case C: another location, but with a 60-minute gap')
solve_travel_case('Riverside Studio', 180)

## Things to try

- In Example 1, add a third `Let's Start 1 #3` group and see which constraint breaks first.
- In Example 1, remove `Room 2` and rerun.
- In Example 1, remove `Mateusz` from `Solo Jazz` eligibility and rerun.
- In Example 2, change durations and see how `AddNoOverlap` moves intervals.
- In Example 3, change `TRAVEL_GAP` from `60` to `30`.

The project solver mostly follows Example 1: build a list of possible candidates, create one Boolean per candidate, then add pairwise conflicts for rooms, instructors, groups, and travel.